# Comparative Analysis of Transfer Learning Strategies on Caltech-101 Using Pre-Trained CNN Architectures

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score
import numpy as np

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [5]:
dataset_path = "/content/drive/MyDrive/caltech-101/101_ObjectCategories" # Corrected path to mounted Google Drive folder

dataset = datasets.ImageFolder(root=dataset_path, transform=transform)

num_classes = 102

## Dataset splitting and Preprocessing

In [6]:
from sklearn.model_selection import train_test_split

SEED = 42

# Get labels and indices
targets = np.array(dataset.targets)
indices = np.arange(len(targets))

# ---- First split: 70% Train, 30% Temp ----
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.3,
    stratify=targets,
    random_state=SEED
)

# ---- Second split: 10% Val, 20% Test ----
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.66,
    stratify=targets[temp_idx],
    random_state=SEED
)

In [7]:
from torch.utils.data import Subset

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))

Train size: 6400
Validation size: 932
Test size: 1812


In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, num_workers = 2, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, num_workers = 2, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, num_workers = 2, shuffle=False)

In [9]:
from collections import Counter

train_labels = targets[train_idx]
val_labels   = targets[val_idx]
test_labels  = targets[test_idx]

print("Train distribution:", Counter(train_labels))
print("Val distribution:", Counter(val_labels))
print("Test distribution:", Counter(test_labels))

Train distribution: Counter({np.int64(6): 560, np.int64(4): 559, np.int64(0): 327, np.int64(2): 304, np.int64(1): 304, np.int64(95): 167, np.int64(3): 140, np.int64(13): 90, np.int64(20): 86, np.int64(56): 80, np.int64(24): 75, np.int64(48): 70, np.int64(14): 69, np.int64(47): 69, np.int64(17): 64, np.int64(51): 62, np.int64(64): 61, np.int64(87): 60, np.int64(93): 60, np.int64(55): 60, np.int64(16): 59, np.int64(91): 59, np.int64(40): 59, np.int64(82): 59, np.int64(58): 57, np.int64(76): 57, np.int64(52): 56, np.int64(59): 55, np.int64(66): 53, np.int64(36): 52, np.int64(94): 52, np.int64(27): 51, np.int64(28): 49, np.int64(35): 48, np.int64(26): 48, np.int64(41): 47, np.int64(42): 47, np.int64(32): 47, np.int64(61): 46, np.int64(54): 45, np.int64(37): 45, np.int64(85): 45, np.int64(34): 45, np.int64(39): 45, np.int64(89): 45, np.int64(80): 44, np.int64(57): 43, np.int64(23): 43, np.int64(101): 42, np.int64(77): 41, np.int64(97): 41, np.int64(88): 41, np.int64(22): 41, np.int64(75): 4

## Model Training and Evaluating functions

In [10]:
def train_model(model, train_loader, val_loader,
                epochs=4, lr=1e-4, save_path="best_model.pth"):

    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    train_losses = []
    val_losses = []
    val_accuracies = []

    best_val_acc = 0

    for epoch in range(epochs):
        # ---- Training ----
        model.train()
        running_loss = 0.0

        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            # --- Debugging checks ---
            if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                print(f"Epoch {epoch+1}, Batch {batch_idx}: NaN or Inf found in model outputs!")
                # Optionally, break or handle this more gracefully
                # For now, we'll let it proceed to see if loss calculation is the exact point
            if labels.min() < 0 or labels.max() >= num_classes:
                print(f"Epoch {epoch+1}, Batch {batch_idx}: Labels out of expected range! Min: {labels.min().item()}, Max: {labels.max().item()}")
            # --- End Debugging checks ---

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)

        # ---- Validation ----
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_loss /= len(val_loader)
        val_acc = correct / total

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(f"Epoch [{epoch+1}/{epochs}] "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f}")

        # ---- Save Best Model ----
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)

    print("Best Validation Accuracy:", best_val_acc)

    return model, train_losses, val_losses, val_accuracies

In [11]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_model(model, test_loader):
    model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')

    print("Test Accuracy:", round(acc, 4))
    print("Test Macro F1:", round(macro_f1, 4))

    return acc, macro_f1

## Resnet-18 Model

In [25]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

model = models.resnet18(pretrained=True)

num_classes = len(dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

In [26]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

In [27]:
model, train_losses, val_losses, val_accuracies = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-3,  
    save_path="resnet18_frozen_best.pth"
)

Epoch [1/8] Train Loss: 2.0575 | Val Loss: 0.8803 | Val Acc: 0.8133
Epoch [2/8] Train Loss: 0.6312 | Val Loss: 0.5355 | Val Acc: 0.8895
Epoch [3/8] Train Loss: 0.3951 | Val Loss: 0.4556 | Val Acc: 0.8916
Epoch [4/8] Train Loss: 0.2848 | Val Loss: 0.3968 | Val Acc: 0.8938
Epoch [5/8] Train Loss: 0.2288 | Val Loss: 0.3624 | Val Acc: 0.9067
Epoch [6/8] Train Loss: 0.1862 | Val Loss: 0.3460 | Val Acc: 0.9034
Epoch [7/8] Train Loss: 0.1547 | Val Loss: 0.3277 | Val Acc: 0.9024
Epoch [8/8] Train Loss: 0.1227 | Val Loss: 0.3174 | Val Acc: 0.9088
Best Validation Accuracy: 0.9087982832618026


In [28]:
model.load_state_dict(torch.load("resnet18_frozen_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("Test Results (ResNet-18 Frozen)")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

Test Accuracy: 0.9056
Test Macro F1: 0.8689

===== Test Results (ResNet-18 Frozen) =====
Test Accuracy : 90.56%
Test Macro F1 : 0.8689


In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

model = models.resnet18(pretrained=True)

num_classes = len(dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

for param in model.layer4.parameters():
    param.requires_grad = True

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s] 


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [57]:
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

In [58]:
model, train_losses, val_losses, val_accuracies = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-4,
    save_path="resnet18_finetuned_best.pth"
)

Epoch [1/8] Train Loss: 1.8433 | Val Loss: 0.7143 | Val Acc: 0.8691
Epoch [2/8] Train Loss: 0.4410 | Val Loss: 0.3807 | Val Acc: 0.9292
Epoch [3/8] Train Loss: 0.1547 | Val Loss: 0.2951 | Val Acc: 0.9410
Epoch [4/8] Train Loss: 0.0652 | Val Loss: 0.2477 | Val Acc: 0.9474
Epoch [5/8] Train Loss: 0.0337 | Val Loss: 0.2465 | Val Acc: 0.9474
Epoch [6/8] Train Loss: 0.0220 | Val Loss: 0.2257 | Val Acc: 0.9496
Epoch [7/8] Train Loss: 0.0154 | Val Loss: 0.2188 | Val Acc: 0.9506
Epoch [8/8] Train Loss: 0.0121 | Val Loss: 0.2120 | Val Acc: 0.9474
Best Validation Accuracy: 0.9506437768240343


In [60]:
model.load_state_dict(torch.load("resnet18_finetuned_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("Test Results (ResNet-18 Finetuned)")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

Test Accuracy: 0.9536
Test Macro F1: 0.9358

===== Test Results (ResNet-18 Finetuned) =====
Test Accuracy : 95.36%
Test Macro F1 : 0.9358


## DenseNet121 Model

In [61]:
from torchvision import models
import torch.nn as nn

model = models.densenet121(pretrained=True)

model.classifier = nn.Linear(model.classifier.in_features, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [62]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [63]:
model, train_losses, val_losses, val_accuracies = train_model(model, train_loader, val_loader, epochs=8, lr=1e-4, save_path="densenet101_frozen_best.pth")

Epoch [1/8] Train Loss: 3.7776 | Val Loss: 3.1105 | Val Acc: 0.3047
Epoch [2/8] Train Loss: 2.7724 | Val Loss: 2.3460 | Val Acc: 0.5032
Epoch [3/8] Train Loss: 2.1370 | Val Loss: 1.8396 | Val Acc: 0.6942
Epoch [4/8] Train Loss: 1.6915 | Val Loss: 1.4599 | Val Acc: 0.7940
Epoch [5/8] Train Loss: 1.3706 | Val Loss: 1.2038 | Val Acc: 0.8380
Epoch [6/8] Train Loss: 1.1367 | Val Loss: 1.0229 | Val Acc: 0.8605
Epoch [7/8] Train Loss: 0.9554 | Val Loss: 0.8869 | Val Acc: 0.8809
Epoch [8/8] Train Loss: 0.8235 | Val Loss: 0.7609 | Val Acc: 0.9002
Best Validation Accuracy: 0.9002145922746781


In [64]:
model.load_state_dict(torch.load("densenet101_frozen_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("Test Results (Densenet-101 Frozen)")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

Test Accuracy: 0.8924
Test Macro F1: 0.8433

===== Test Results (Densenet-101 Frozen) =====
Test Accuracy : 89.24%
Test Macro F1 : 0.8433


In [65]:
from torchvision import models
import torch.nn as nn

model = models.densenet121(pretrained=True)

model.classifier = nn.Linear(model.classifier.in_features, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

for param in model.features.denseblock4.parameters():
    param.requires_grad = True

for param in model.features.norm5.parameters():
    param.requires_grad = True


model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [66]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [67]:
model, train_losses, val_losses, val_accuracies = train_model(model, train_loader, val_loader, epochs=8, lr=1e-4, save_path="densenet101_finetuned_best.pth")

Epoch [1/8] Train Loss: 2.5723 | Val Loss: 1.3899 | Val Acc: 0.7672
Epoch [2/8] Train Loss: 1.0395 | Val Loss: 0.6608 | Val Acc: 0.9099
Epoch [3/8] Train Loss: 0.5194 | Val Loss: 0.4118 | Val Acc: 0.9388
Epoch [4/8] Train Loss: 0.2938 | Val Loss: 0.3042 | Val Acc: 0.9464
Epoch [5/8] Train Loss: 0.1815 | Val Loss: 0.2610 | Val Acc: 0.9506
Epoch [6/8] Train Loss: 0.1186 | Val Loss: 0.2284 | Val Acc: 0.9539
Epoch [7/8] Train Loss: 0.0794 | Val Loss: 0.2089 | Val Acc: 0.9549
Epoch [8/8] Train Loss: 0.0559 | Val Loss: 0.1989 | Val Acc: 0.9528
Best Validation Accuracy: 0.9549356223175965


In [68]:
model.load_state_dict(torch.load("densenet101_finetuned_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("Test Results (Densenet-101 Finetuned)")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

Test Accuracy: 0.947
Test Macro F1: 0.9235

===== Test Results (Densenet-101 Finetuned) =====
Test Accuracy : 94.70%
Test Macro F1 : 0.9235


## MobileNet V2 Model

In [12]:
from torchvision import models
import torch.nn as nn

model = models.mobilenet_v2(pretrained=True)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier[1].parameters():
    param.requires_grad = True

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 105MB/s] 


In [13]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

In [14]:
model, train_losses, val_losses, val_accuracies = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-3,
    save_path="mobilenetv2_frozen_best.pth"
)

Epoch [1/8] Train Loss: 1.6243 | Val Loss: 0.5955 | Val Acc: 0.8820
Epoch [2/8] Train Loss: 0.4440 | Val Loss: 0.3975 | Val Acc: 0.9088
Epoch [3/8] Train Loss: 0.2753 | Val Loss: 0.3541 | Val Acc: 0.9120
Epoch [4/8] Train Loss: 0.2059 | Val Loss: 0.3284 | Val Acc: 0.9120
Epoch [5/8] Train Loss: 0.1556 | Val Loss: 0.3275 | Val Acc: 0.9174
Epoch [6/8] Train Loss: 0.1210 | Val Loss: 0.2989 | Val Acc: 0.9195
Epoch [7/8] Train Loss: 0.0993 | Val Loss: 0.2996 | Val Acc: 0.9152
Epoch [8/8] Train Loss: 0.0891 | Val Loss: 0.2880 | Val Acc: 0.9163
Best Validation Accuracy: 0.9195278969957081


In [15]:
model.load_state_dict(torch.load("mobilenetv2_frozen_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("MobileNetV2 Frozen Results")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

Test Accuracy: 0.9128
Test Macro F1: 0.882

MobileNetV2 Frozen Results
Test Accuracy : 91.28%
Test Macro F1 : 0.8820


In [16]:
from torchvision import models
import torch.nn as nn

model = models.mobilenet_v2(pretrained=True)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier[1].parameters():
    param.requires_grad = True

for param in model.features[-2:].parameters():
    param.requires_grad = True

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [17]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)

In [18]:
model, train_losses, val_losses, val_accuracies = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-4,
    save_path="mobilenetv2_finetuned_best.pth"
)

Epoch [1/8] Train Loss: 2.5225 | Val Loss: 1.3818 | Val Acc: 0.7436
Epoch [2/8] Train Loss: 1.0644 | Val Loss: 0.7252 | Val Acc: 0.8734
Epoch [3/8] Train Loss: 0.6020 | Val Loss: 0.4836 | Val Acc: 0.9217
Epoch [4/8] Train Loss: 0.3969 | Val Loss: 0.3934 | Val Acc: 0.9303
Epoch [5/8] Train Loss: 0.2866 | Val Loss: 0.3395 | Val Acc: 0.9356
Epoch [6/8] Train Loss: 0.2188 | Val Loss: 0.2937 | Val Acc: 0.9388
Epoch [7/8] Train Loss: 0.1689 | Val Loss: 0.2804 | Val Acc: 0.9399
Epoch [8/8] Train Loss: 0.1313 | Val Loss: 0.2621 | Val Acc: 0.9410
Best Validation Accuracy: 0.9409871244635193


In [19]:
model.load_state_dict(torch.load("mobilenetv2_finetuned_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("MobileNetV2 Fine-Tuned Results")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")

Test Accuracy: 0.9316
Test Macro F1: 0.9029

MobileNetV2 Fine-Tuned Results
Test Accuracy : 93.16%
Test Macro F1 : 0.9029


In [69]:
transform = transforms.Compose([
    transforms.Resize((260, 260)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## EfficientNet-B2 Model with Change in Dimensions for the image

In [70]:
dataset_path = "/content/drive/MyDrive/caltech-101/101_ObjectCategories" # Corrected path to mounted Google Drive folder

dataset = datasets.ImageFolder(root=dataset_path, transform=transform)

num_classes = 102

In [71]:
from sklearn.model_selection import train_test_split

SEED = 42

# Get labels and indices
targets = np.array(dataset.targets)
indices = np.arange(len(targets))

# ---- First split: 70% Train, 30% Temp ----
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.3,
    stratify=targets,
    random_state=SEED
)

# ---- Second split: 10% Val, 20% Test ----
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.66,
    stratify=targets[temp_idx],
    random_state=SEED
)

In [72]:
from torch.utils.data import Subset

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))

Train size: 6400
Validation size: 932
Test size: 1812


In [73]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, num_workers = 2, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, num_workers = 2, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, num_workers = 2, shuffle=False)

In [74]:
from collections import Counter

train_labels = targets[train_idx]
val_labels   = targets[val_idx]
test_labels  = targets[test_idx]

print("Train distribution:", Counter(train_labels))
print("Val distribution:", Counter(val_labels))
print("Test distribution:", Counter(test_labels))

Train distribution: Counter({np.int64(6): 560, np.int64(4): 559, np.int64(0): 327, np.int64(2): 304, np.int64(1): 304, np.int64(95): 167, np.int64(3): 140, np.int64(13): 90, np.int64(20): 86, np.int64(56): 80, np.int64(24): 75, np.int64(48): 70, np.int64(14): 69, np.int64(47): 69, np.int64(17): 64, np.int64(51): 62, np.int64(64): 61, np.int64(87): 60, np.int64(93): 60, np.int64(55): 60, np.int64(16): 59, np.int64(91): 59, np.int64(40): 59, np.int64(82): 59, np.int64(58): 57, np.int64(76): 57, np.int64(52): 56, np.int64(59): 55, np.int64(66): 53, np.int64(36): 52, np.int64(94): 52, np.int64(27): 51, np.int64(28): 49, np.int64(35): 48, np.int64(26): 48, np.int64(41): 47, np.int64(42): 47, np.int64(32): 47, np.int64(61): 46, np.int64(54): 45, np.int64(37): 45, np.int64(85): 45, np.int64(34): 45, np.int64(39): 45, np.int64(89): 45, np.int64(80): 44, np.int64(57): 43, np.int64(23): 43, np.int64(101): 42, np.int64(77): 41, np.int64(97): 41, np.int64(88): 41, np.int64(22): 41, np.int64(75): 4

In [75]:
import torch
import torch.nn as nn
from torchvision import models

model = models.efficientnet_b2(pretrained=True)

num_classes = len(dataset.classes)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    num_classes
)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B2_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 103MB/s] 


In [76]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [77]:
model, train_losses, val_losses, val_accuracies = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-3,
    save_path="efficientnet_b2_frozen_best.pth"
)

Epoch [1/8] Train Loss: 1.8722 | Val Loss: 0.7877 | Val Acc: 0.8970
Epoch [2/8] Train Loss: 0.5101 | Val Loss: 0.4762 | Val Acc: 0.9142
Epoch [3/8] Train Loss: 0.3151 | Val Loss: 0.3892 | Val Acc: 0.9185
Epoch [4/8] Train Loss: 0.2411 | Val Loss: 0.3350 | Val Acc: 0.9281
Epoch [5/8] Train Loss: 0.1911 | Val Loss: 0.3119 | Val Acc: 0.9313
Epoch [6/8] Train Loss: 0.1565 | Val Loss: 0.3111 | Val Acc: 0.9281
Epoch [7/8] Train Loss: 0.1286 | Val Loss: 0.2887 | Val Acc: 0.9324
Epoch [8/8] Train Loss: 0.1137 | Val Loss: 0.2817 | Val Acc: 0.9303
Best Validation Accuracy: 0.9324034334763949


In [78]:
model.load_state_dict(torch.load("efficientnet_b2_frozen_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("\n===== Test Results (Efficientnet-B2 Frozen) =====")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")
print("===========================================")

Test Accuracy: 0.92
Test Macro F1: 0.8902

===== Test Results (Efficientnet-B2 Frozen) =====
Test Accuracy : 92.00%
Test Macro F1 : 0.8902


In [80]:
model = models.efficientnet_b2(pretrained=True)

num_classes = len(dataset.classes)

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    num_classes
)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

for param in model.features[-1].parameters():
    param.requires_grad = True

for param in model.features[-2].parameters():
    param.requires_grad = True

model = model.to(device)

In [81]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [82]:
model, train_losses, val_losses, val_accuracies = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=8,
    lr=1e-4,
    save_path="efficientnet_b2_finetuned_best.pth"
)

Epoch [1/8] Train Loss: 2.8790 | Val Loss: 1.5943 | Val Acc: 0.7790
Epoch [2/8] Train Loss: 1.1037 | Val Loss: 0.6075 | Val Acc: 0.9120
Epoch [3/8] Train Loss: 0.5136 | Val Loss: 0.3587 | Val Acc: 0.9249
Epoch [4/8] Train Loss: 0.3244 | Val Loss: 0.2876 | Val Acc: 0.9399
Epoch [5/8] Train Loss: 0.2172 | Val Loss: 0.2432 | Val Acc: 0.9399
Epoch [6/8] Train Loss: 0.1568 | Val Loss: 0.2334 | Val Acc: 0.9431
Epoch [7/8] Train Loss: 0.1321 | Val Loss: 0.2105 | Val Acc: 0.9485
Epoch [8/8] Train Loss: 0.0996 | Val Loss: 0.2014 | Val Acc: 0.9517
Best Validation Accuracy: 0.9517167381974249


In [83]:
model.load_state_dict(torch.load("efficientnet_b2_finetuned_best.pth"))

test_acc, test_f1 = evaluate_model(model, test_loader)

print("\n===== Test Results (Efficientnet-B2 Frozen) =====")
print(f"Test Accuracy : {test_acc*100:.2f}%")
print(f"Test Macro F1 : {test_f1:.4f}")
print("===========================================")

Test Accuracy: 0.9443
Test Macro F1: 0.9186

===== Test Results (Efficientnet-B2 Frozen) =====
Test Accuracy : 94.43%
Test Macro F1 : 0.9186
